# Activity 2 — RNN / LSTM on MNIST
## Module 07 — Deep Learning in Practice | ISY503

MNIST images are 28×28 pixels. For an RNN we treat each image as a **sequence of 28 rows**, each row being a vector of 28 pixel values.
The LSTM processes one row at a time, building up a hidden state — at the end of the 28th row it classifies the digit.

Architecture: `LSTM(128) → Dropout → LSTM(128) → Dropout → Dense(32) → Dropout → Dense(10, softmax)`

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
mnist = tf.keras.datasets.mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize pixel values to [0, 1]
x_train = tf.keras.utils.normalize(x_train, axis=1)
x_test  = tf.keras.utils.normalize(x_test,  axis=1)

# Shape: (60000, 28, 28) — 28 time steps, each with 28 features (pixel values per row)
# No Flatten needed — LSTM consumes the sequence directly
print("Train shape:", x_train.shape)
print("Test shape: ", x_test.shape)

In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),

    # LSTM layer 1: return_sequences=True passes hidden state at every time step to the next LSTM
    tf.keras.layers.LSTM(128, activation='relu', return_sequences=True),
    tf.keras.layers.Dropout(0.2),  # drop 20% of neurons to reduce overfitting

    # LSTM layer 2: return_sequences=False (default) — only the final hidden state feeds forward
    tf.keras.layers.LSTM(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(10, activation='softmax'),  # 10 digit classes
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3, weight_decay=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    x_train, y_train,
    epochs=3,
    validation_data=(x_test, y_test)
)

In [ ]:
val_loss, val_acc = model.evaluate(x_test, y_test)
print(f"Test loss:     {val_loss:.4f}")
print(f"Test accuracy: {val_acc:.4f}")

In [ ]:
plt.plot(history.history['accuracy'],     label='train accuracy')
plt.plot(history.history['val_accuracy'], label='val accuracy')
plt.title('LSTM — Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()